In [0]:
# 00_generate_data: synthetic bank transactions with duplicates and bad rows
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bank")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bank.raw")

import random, datetime
import pandas as pd

random.seed(42)
accounts = [f"ACC{n:03d}" for n in range(1, 21)]
start = datetime.date(2026, 10, 1)
rows = []
for i in range(1, 1001):
    txn_date = start + datetime.timedelta(days=random.randint(0, 29))
    load_ts = datetime.datetime.combine(txn_date, datetime.time(2, 0)) + datetime.timedelta(days=1)
    rows.append({
        "txn_id": f"T{i:05d}",
        "account_id": random.choice(accounts),
        "txn_date": txn_date.isoformat(),
        "amount": round(random.uniform(-5000, 8000), 2),
        "load_ts": load_ts.isoformat(sep=" "),
    })

# pipeline re-runs: 50 transactions loaded again 6 hours later (duplicates)
for r in random.sample(rows, 50):
    dup = dict(r)
    dup["load_ts"] = (datetime.datetime.fromisoformat(r["load_ts"]) + datetime.timedelta(hours=6)).isoformat(sep=" ")
    rows.append(dup)

# bad records for data quality checks
rows.append({"txn_id": "T99001", "account_id": None, "txn_date": "2026-10-15", "amount": 100.0, "load_ts": "2026-10-16 02:00:00"})
rows.append({"txn_id": "T99002", "account_id": "ACC005", "txn_date": "2026-10-15", "amount": None, "load_ts": "2026-10-16 02:00:00"})

df = pd.DataFrame(rows)
path = "/Volumes/workspace/bank/raw/transactions"
dbutils.fs.mkdirs(path)
df.to_csv(f"{path}/batch_001.csv", index=False)
print(len(df), "rows written")
display(spark.read.option("header", True).csv(path).limit(10))

1052 rows written


txn_id,account_id,txn_date,amount,load_ts
T00001,ACC004,2026-10-21,-4674.86,2026-10-22 02:00:00
T00002,ACC008,2026-10-09,-2098.26,2026-10-10 02:00:00
T00003,ACC004,2026-10-24,3797.09,2026-10-25 02:00:00
T00004,ACC018,2026-10-29,-3869.8,2026-10-30 02:00:00
T00005,ACC002,2026-10-14,-4612.64,2026-10-15 02:00:00
T00006,ACC008,2026-10-07,1569.62,2026-10-08 02:00:00
T00007,ACC018,2026-10-01,-2415.11,2026-10-02 02:00:00
T00008,ACC018,2026-10-21,453.76,2026-10-22 02:00:00
T00009,ACC019,2026-10-15,-1383.52,2026-10-16 02:00:00
T00010,ACC001,2026-10-28,4864.5,2026-10-29 02:00:00


In [0]:
import pandas as pd
path = "/Volumes/workspace/bank/raw/transactions"
batch2 = pd.DataFrame([
    # two new transactions for 31 Oct
    {"txn_id": "T01001", "account_id": "ACC001", "txn_date": "2026-10-31", "amount": 2500.00, "load_ts": "2026-11-01 02:00:00"},
    {"txn_id": "T01002", "account_id": "ACC002", "txn_date": "2026-10-31", "amount": -750.50, "load_ts": "2026-11-01 02:00:00"},
    # correction: source system re-sends T00001 with a corrected amount
    {"txn_id": "T00001", "account_id": "ACC004", "txn_date": "2026-10-21", "amount": -4600.00, "load_ts": "2026-11-01 02:00:00"},
])
batch2.to_csv(f"{path}/batch_002.csv", index=False)
print("batch_002 written")

batch_002 written
